# CDR-MLC: paper-aligned implementation

The implementation is in `cdr_mlc.py`, tested independently of notebook state.
See `README.md` for the leakage audit, explicit implementation choices, and data limitations.
Cluster IDs are **not automatically physical congestion levels**. No target labels are read at inference.
Old outputs were removed because they were produced with label leakage / oracle options.


In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if (ROOT / 'CDR_MLC').is_dir():
    ROOT = ROOT / 'CDR_MLC'
if not (ROOT / 'cdr_mlc.py').is_file():
    raise RuntimeError('Open this notebook from the repository root or CDR_MLC directory')
sys.path.insert(0, str(ROOT))
from cdr_mlc import CDRMLC, Config, trend_features, evaluate
from run_cdr_mlc import run_experiment


## Choose the experiment
Scenarios 1–3 train on a single observed congestion level. The router still learns 3 geometric clusters; this does not prove it has learned all 3 physical levels.
Scenarios 4–5 use all short levels versus the long capture. Confirm real capture order before interpreting temporal results.


In [ ]:
SCENARIO = 1
levels = [ROOT / f'DATASETS/CDR-MLC/scale_1/Short/level_{i}.csv' for i in (1, 2, 3)]
long_file = ROOT / 'DATASETS/CDR-MLC/scale_1/Long/CDR-MLC-notShuffle.csv'
scenarios = {
    1: ([levels[0]], [levels[1]]),
    2: ([levels[0]], [levels[2]]),
    3: ([levels[1]], [levels[2]]),
    4: (levels, [long_file]),
    5: ([long_file], levels),
}
train_files, test_files = scenarios[SCENARIO]
report = run_experiment(train_files, test_files,
                        ROOT / f'results/scenario_{SCENARIO}',
                        config=Config(), compare_agglomerative=True)
report['metrics']


In [ ]:
# Inspect missing class coverage and routing rather than moving samples between clusters.
report['training_diagnostics']['cluster_by_class'], report['test_diagnostics']['cluster_counts']


## Stateful deployment (unlabeled input)
Load `model.joblib` only from trusted files. Use `model.stream().predict_one(sample)` in observation order. Supply the same capture-ID field used in training (`_capture_id` in the runner); change it only at a real capture boundary. Do not supply class or congestion labels to choose a model.
